In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess
from matplotlib.lines import Line2D

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["font.size"] = 12

# ============================================================
# 1. Read
# ============================================================

file = "/Users/apple/Downloads/benchmarks/es/test_555_rep1.barcodes.tsv"

df = pd.read_csv(file, sep='\t')

print(df.columns.tolist())

print("\nNumber of barcodes:")
print(len(df))

# ============================================================
# 2. Define calling groups from barcode format
#
# xxx#large  -> large
# xxx#small  -> small
# EMPTYxxxx  -> EMPTY
# ============================================================

barcode = df["barcode"].astype(str)

df["plot_group"] = "Other"

df.loc[barcode.str.endswith('#large'), 'plot_group'] = 'large'

df.loc[barcode.str.endswith('#small') | barcode.str.startswith('EMPTY'), 'plot_group'] = 'small_EMPTY'

print("\nPlot groups:")
print(df["plot_group"].value_counts())

# ============================================================
# 3. Sort by original rank
#
# The original rank may range from 2 to 6015.
# We use it only to determine the ordering of these 3000
# barcodes, then generate a new rank from 1 to 3000.
# ============================================================

plot_df = df[df['umis'] > 0].copy()

plot_df = plot_df.sort_values('rank').reset_index(drop=True)

plot_df['plot_rank'] = np.arange(1, len(plot_df) + 1)

print("\nNew plot rank:")
print("min =", plot_df["plot_rank"].min())
print("max =", plot_df["plot_rank"].max())

# ============================================================
# 4. Transform variables
# ============================================================

# y axis
plot_df['log_umi'] = np.log10(plot_df['umis'])

# x axis
#
# 0.5 = sqrt
# 0.75 = more expanded
# 1.0 = linear
#
plot_df['x_rank'] = plot_df['plot_rank'] ** 0.75

# ============================================================
# 5. LOWESS function
# ============================================================

def add_lowess(
    data,
    color="black",
    frac=0.025
):

    if len(data) < 10:
        return

    d = data.sort_values("x_rank")

    fit = lowess(
        endog=d["log_umi"].values,
        exog=d["x_rank"].values,
        frac=frac,
        it=3,
        return_sorted=True
    )

    ax.plot(
        fit[:, 0],
        fit[:, 1],
        color=color,
        linewidth=1,
        zorder=5
    )

# ============================================================
# 6. Create figure
# ============================================================

fig, ax = plt.subplots(figsize=(5.5, 4.5))

# ============================================================
# 7. Other barcodes
# ============================================================

other = plot_df[plot_df['plot_group'] == 'Other']

ax.scatter(other['x_rank'], other['log_umi'], s=14, alpha=0.2, color='0.75', linewidths=0, rasterized=True)

# ============================================================
# 8. Large cells
# ============================================================

large = plot_df[plot_df['plot_group'] == 'large']

ax.scatter(large['x_rank'], large['log_umi'], s=14, alpha=0.65, color='C0', linewidths=0, rasterized=True)

# ============================================================
# 9. Small + EMPTY
# ============================================================

small_empty = plot_df[plot_df['plot_group'] == 'small_EMPTY']

ax.scatter(
    small_empty["x_rank"],
    small_empty["log_umi"],
    s=14,
    alpha=0.65,
    color="#E9A3A3",
    linewidths=0,
    rasterized=True
)

# ============================================================
# 10. LOWESS
# ============================================================

add_lowess(large, color='black', frac=0.025)

add_lowess(small_empty, color='black', frac=0.025)

# ============================================================
# 11. X-axis ticks
# ============================================================

rank_ticks = np.array([1, 10, 100, 500, 1000, 1500, 2000, 2500, 3000])

rank_ticks = rank_ticks[rank_ticks <= plot_df['plot_rank'].max()]

ax.set_xticks(rank_ticks ** 0.75)

ax.set_xticklabels([str(x) for x in rank_ticks])

# ============================================================
# 12. Labels
# ============================================================

ax.set_xlabel('Barcode rank')

ax.set_ylabel('$\\log_{10}$(UMIs)')

# ============================================================
# 13. Style
# ============================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(direction='out', length=4, width=1)

# ============================================================
# 14. Legend
# ============================================================

legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color="C0",
        linestyle="None",
        markersize=5,
        label="Large cells"
    ),
    Line2D(
        [0], [0],
        marker="o",
        color="#E9A3A3",
        linestyle="None",
        markersize=5,
        label="Small cells / Empty mixture"
    ),
    Line2D(
        [0], [0],
        color="black",
        linewidth=1,
        label="Fitting"
    )
]

ax.legend(handles=legend_handles, frameon=False, loc='best')

# ============================================================
# 15. Layout
# ============================================================

plt.tight_layout()

plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["font.size"] = 12

# ============================================================
# 1. Read counts.tsv
# ============================================================

counts_file = '/Users/apple/Downloads/benchmarks/es/counts.tsv'

df = pd.read_csv(counts_file, sep='\t', dtype={'barcode': str})

print("counts.tsv columns:")
print(df.columns.tolist())

print(f'\n总 barcode 数: {len(df)}')
print(f'总 UMI 数: {df["umi_count"].sum():,}')
print(f"UMI 范围: {df['umi_count'].min()} ~ {df['umi_count'].max()}")

# ============================================================
# 2. Sort by UMI count and assign rank
# ============================================================

df_sorted = df.sort_values('umi_count', ascending=False).reset_index(drop=True)

df_sorted['rank'] = np.arange(1, len(df_sorted) + 1)

# 确保 barcode 是字符串
df_sorted['barcode'] = df_sorted['barcode'].astype(str)

# ============================================================
# 3. Read run1.barcodes.tsv
# ============================================================

barcode_file = '/Users/apple/Downloads/benchmarks/es/run2.barcodes.tsv'

barcode_df = pd.read_csv(barcode_file, sep='\t', dtype={'barcode': str})

barcode_df['barcode'] = barcode_df['barcode'].astype(str)

print(f'\nrun1 barcode 数: {len(barcode_df)}')

print(f"cell barcode 数: {(barcode_df['call'] == 'cell').sum()}")

# ============================================================
# 4. Check barcode overlap
# ============================================================

counts_barcodes = set(df_sorted['barcode'])

run1_barcodes = set(barcode_df['barcode'])

overlap = counts_barcodes & run1_barcodes

print(f'\nbarcode overlap: {len(overlap)}')

# ============================================================
# 5. Identify cell barcodes
# ============================================================

cell_barcodes = set(barcode_df.loc[barcode_df['call'] == 'cell', 'barcode'])

# 在 counts.tsv 中标记 cell
df_sorted['is_cell'] = df_sorted['barcode'].isin(cell_barcodes)

print(f"匹配到的 cell barcode 数: {df_sorted['is_cell'].sum()}")

# ============================================================
# 6. Define log-spaced bins
# ============================================================

max_rank = len(df_sorted)

n_bins = 30

bins = np.logspace(np.log10(1), np.log10(max_rank + 1), n_bins + 1)

# ============================================================
# 7. Assign run1 barcodes to rank bins
# ============================================================

barcode_df['rank_bin'] = pd.cut(barcode_df['rank'], bins=bins, include_lowest=True, right=False)

# ============================================================
# 8. Calculate cell percentage per bin
# ============================================================

bin_pct = (
    barcode_df
    .groupby(
        'rank_bin',
        observed=True
    )
    .agg(
        total=('call', 'size'),
        cells=(
            'call',
            lambda x: (
                x == 'cell'
            ).sum()
        )
    )
)

bin_pct['cell_pct'] = bin_pct['cells'] / bin_pct['total'] * 100

# ============================================================
# 9. Exact bin boundaries
# ============================================================

bin_left = np.array([interval.left for interval in bin_pct.index])

bin_right = np.array([interval.right for interval in bin_pct.index])

# log scale 下的几何中心
bin_center = np.sqrt(bin_left * bin_right)

# 真实 bin 宽度
bin_width = bin_right - bin_left

# ============================================================
# 10. Plot
# ============================================================

fig, ax = plt.subplots(figsize=(7, 6))

# ============================================================
# Colors
# ============================================================

grey = '0.65'
blue = 'tab:blue'

# ============================================================
# 11. All barcode scatter — grey
# ============================================================

ax.scatter(
    df_sorted['rank'],
    df_sorted['umi_count'],
    s=16,
    color=grey,
    edgecolors='none',
    zorder=2,
    rasterized=True
)

# ============================================================
# 12. Cell barcode scatter — blue
# ============================================================

cell_points = df_sorted[df_sorted['is_cell']]

ax.scatter(
    cell_points['rank'],
    cell_points['umi_count'],
    s=10,
    color=blue,
    edgecolors='none',
    zorder=3,
    rasterized=True
)

# ============================================================
# 13. Left axis
# ============================================================

ax.set_xlabel('Barcode Rank', fontsize=19)

ax.set_ylabel('Total UMI Count', fontsize=19)

ax.set_xscale('log')
ax.set_yscale('log')

ax.tick_params(axis='both', labelsize=18)

# ============================================================
# 14. Right axis
# ============================================================

ax2 = ax.twinx()

# ============================================================
# 15. Cell percentage bin chart
# ============================================================

ax2.bar(
    bin_center,
    bin_pct['cell_pct'],
    width=bin_width,
    align='center',
    color=blue,
    alpha=0.35,
    edgecolor='none',
    linewidth=0,
    zorder=1
)

# ============================================================
# 16. Right axis settings
# ============================================================

ax2.set_ylabel('cell called(%)', fontsize=19)

ax2.set_ylim(0, 105)

ax2.tick_params(axis='y', labelsize=18)

ax2.set_xscale('log')

ax2.set_xlim(ax.get_xlim())

# ============================================================
# 17. Layout
# ============================================================

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["font.size"] = 12

# ============================================================
# Raw results
#
# 每一行 = 一个 replicate
# 每一列 = large / EMPTY / small
#
# 直接按照程序输出填写
# ============================================================

replicates = ["rep1", "rep2", "rep3"]

# ------------------------------------------------------------
# ModuleDrops
# cell_call (col11 == cell)
# ------------------------------------------------------------

module_drops = np.array([[500, 5, 500], [500, 3, 500], [500, 2, 500]], dtype=float)

# ------------------------------------------------------------
# EmptyDrops (non-rRNA)
# padj_nonrrna (col18 < 0.01)
# ------------------------------------------------------------

emptydrops_nonrrna = np.array([[19, 0, 1], [18, 0, 2], [18, 0, 1]], dtype=float)

# ------------------------------------------------------------
# EmptyDrops
# padj_total (col20 < 0.01)
# ------------------------------------------------------------

emptydrops = np.array([[496, 1, 485], [496, 1, 486], [496, 0, 481]], dtype=float)

# ------------------------------------------------------------
# Kneedle
# ------------------------------------------------------------

kneedle = np.array([[52, 0, 0], [52, 0, 0], [52, 0, 0]], dtype=float)

# ============================================================
# Combine into the format used by plotting code
# ============================================================

# Row = method
# Column = replicate

methods = ['ModuleDrops', 'Knee', 'EmptyDrops', 'EmptyDrops\n(non-rRNA)']

large = np.array([module_drops[:, 0], kneedle[:, 0], emptydrops[:, 0], emptydrops_nonrrna[:, 0]])

small = np.array([module_drops[:, 2], kneedle[:, 1], emptydrops[:, 2], emptydrops_nonrrna[:, 2]])

empty = np.array([module_drops[:, 1], kneedle[:, 2], emptydrops[:, 1], emptydrops_nonrrna[:, 1]])

n_large = 500
n_small = 500
n_empty = 500

# ============================================================
# Mean ± SEM
# ============================================================

# Large cells
large_mean = large.mean(axis=1) / n_large
large_sem = large.std(axis=1, ddof=1) / np.sqrt(large.shape[1]) / n_large

# Small cells
small_mean = small.mean(axis=1) / n_small
small_sem = small.std(axis=1, ddof=1) / np.sqrt(small.shape[1]) / n_small

# Empty droplets / FDR
fdr_mean = empty.mean(axis=1) / n_empty
fdr_sem = empty.std(axis=1, ddof=1) / np.sqrt(empty.shape[1]) / n_empty

# ============================================================
# Plot parameters
# ============================================================

# Method-to-method spacing
x = np.arange(len(methods)) * 0.5

# Width of each bar
bar_width = 0.10

# Distance between centers of the 3 bars
bar_spacing = 0.105

# Replicate jitter
jitter = np.array([-0.01, 0, 0.01])

# ============================================================
# Figure
# ============================================================

fig, ax1 = plt.subplots(figsize=(4.6, 5))

# ============================================================
# Recall: Large cells
# ============================================================

bars_large = ax1.bar(
    x - bar_spacing,
    large_mean,
    bar_width,
    yerr=large_sem,
    error_kw={
        "elinewidth": 0.8,
        "capsize": 2.5,
        "capthick": 0.8
    },
    label="Large cells"
)

# ============================================================
# Recall: Small cells
# ============================================================

bars_small = ax1.bar(
    x,
    small_mean,
    bar_width,
    yerr=small_sem,
    error_kw={
        "elinewidth": 0.8,
        "capsize": 2.5,
        "capthick": 0.8
    },
    label="Small cells"
)

ax1.set_ylabel("Recall")
ax1.set_ylim(0, 1.28)

# ============================================================
# FDR: right axis
# ============================================================

ax2 = ax1.twinx()

bars_fdr = ax2.bar(
    x + bar_spacing,
    fdr_mean,
    bar_width,
    yerr=fdr_sem,
    error_kw={
        "elinewidth": 0.8,
        "capsize": 2.5,
        "capthick": 0.8
    },
    color="red",
    label="FDR"
)

ax2.set_ylabel("False discovery rate")
ax2.set_ylim(0, 0.05)

# ============================================================
# Percentage labels
# ============================================================

def add_labels(ax, bars, values, offset):

    for bar, value in zip(bars, values):

        height = bar.get_height()

        if value == 0:
            label = "0%"
        else:
            label = f"{value * 100:.1f}%"

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + offset,
            label,
            ha="center",
            va="bottom",
            fontsize=7
        )

# Large
add_labels(ax1, bars_large, large_mean, offset=0.05)

# Small
add_labels(ax1, bars_small, small_mean, offset=0.13)

# FDR
add_labels(ax2, bars_fdr, fdr_mean, offset=0.005)

# ============================================================
# Replicate markers
# ============================================================

markers = ['o', 'o', 'o', 'o']

for i in range(len(methods)):

    # --------------------------------------------------------
    # Large
    # --------------------------------------------------------

    ax1.scatter(
        x[i] - bar_spacing + jitter,
        large[i] / n_large,
        marker=markers[i],
        facecolors="none",
        edgecolors="black",
        s=20,
        linewidths=0.5,
        zorder=10
    )

    # --------------------------------------------------------
    # Small
    # --------------------------------------------------------

    ax1.scatter(
        x[i] + jitter,
        small[i] / n_small,
        marker=markers[i],
        facecolors="none",
        edgecolors="black",
        s=20,
        linewidths=0.5,
        zorder=10
    )

    # --------------------------------------------------------
    # FDR
    # --------------------------------------------------------

    ax2.scatter(
        x[i] + bar_spacing + jitter,
        empty[i] / n_empty,
        marker=markers[i],
        facecolors="none",
        edgecolors="black",
        s=20,
        linewidths=0.5,
        zorder=10
    )

# ============================================================
# X axis
# ============================================================

ax1.set_xticks(x)

ax1.set_xticklabels(methods, rotation=20, ha='right', fontsize=8)

ax1.set_title('Large/Small/Empty 500/500/500', fontsize=9, pad=4)

# ============================================================
# Combined legend
# ============================================================

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    frameon=False,
    loc='upper center',
    ncol=3,
    fontsize=8,
    columnspacing=1.0,
    handletextpad=0.4,
    bbox_to_anchor=(0.5, 1.0)
)

# ============================================================
# Layout
# ============================================================

plt.tight_layout()

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["font.size"] = 12

# ============================================================
# Raw results
#
# 每一行 = 一个 replicate
# 每一列 = large / EMPTY / small
#
# 直接按照程序输出填写
# ============================================================

replicates = ["5125_rep1", "5125_rep2", "5125_rep3"]

# ------------------------------------------------------------
# ModuleDrops
# cell_call (col11 == cell)
# ------------------------------------------------------------

module_drops = np.array([[500, 2, 1000], [500, 1, 1000], [500, 0, 1000]], dtype=float)

# ------------------------------------------------------------
# EmptyDrops (non-rRNA)
# padj_nonrrna (col18 < 0.01)
# ------------------------------------------------------------

emptydrops_nonrrna = np.array([[18, 0, 0], [18, 0, 1], [19, 0, 3]], dtype=float)

# ------------------------------------------------------------
# EmptyDrops
# padj_total (col20 < 0.01)
# ------------------------------------------------------------

emptydrops = np.array([[497, 1, 981], [497, 0, 972], [497, 1, 979]], dtype=float)

# ------------------------------------------------------------
# Kneedle
# ------------------------------------------------------------

kneedle = np.array([[60, 0, 0], [60, 0, 0], [60, 0, 0]], dtype=float)

# ============================================================
# Combine into the format used by plotting code
# ============================================================

# Row = method
# Column = replicate

methods = ['ModuleDrops', 'Knee', 'EmptyDrops', 'EmptyDrops\n(non-rRNA)']

large = np.array([module_drops[:, 0], kneedle[:, 0], emptydrops[:, 0], emptydrops_nonrrna[:, 0]])

small = np.array([module_drops[:, 2], kneedle[:, 1], emptydrops[:, 2], emptydrops_nonrrna[:, 2]])

empty = np.array([module_drops[:, 1], kneedle[:, 2], emptydrops[:, 1], emptydrops_nonrrna[:, 1]])

n_large = 500
n_small = 1000
n_empty = 250

# ============================================================
# Mean ± SEM
# ============================================================

# Large cells
large_mean = large.mean(axis=1) / n_large
large_sem = large.std(axis=1, ddof=1) / np.sqrt(large.shape[1]) / n_large

# Small cells
small_mean = small.mean(axis=1) / n_small
small_sem = small.std(axis=1, ddof=1) / np.sqrt(small.shape[1]) / n_small

# Empty droplets / FDR
fdr_mean = empty.mean(axis=1) / n_empty
fdr_sem = empty.std(axis=1, ddof=1) / np.sqrt(empty.shape[1]) / n_empty

# ============================================================
# Plot parameters
# ============================================================

# Method-to-method spacing
x = np.arange(len(methods)) * 0.5

# Width of each bar
bar_width = 0.10

# Distance between centers of the 3 bars
bar_spacing = 0.105

# Replicate jitter
jitter = np.array([-0.01, 0, 0.01])

# ============================================================
# Figure
# ============================================================

fig, ax1 = plt.subplots(figsize=(4.6, 5))

# ============================================================
# Recall: Large cells
# ============================================================

bars_large = ax1.bar(
    x - bar_spacing,
    large_mean,
    bar_width,
    yerr=large_sem,
    error_kw={
        "elinewidth": 0.8,
        "capsize": 2.5,
        "capthick": 0.8
    },
    label="Large cells"
)

# ============================================================
# Recall: Small cells
# ============================================================

bars_small = ax1.bar(
    x,
    small_mean,
    bar_width,
    yerr=small_sem,
    error_kw={
        "elinewidth": 0.8,
        "capsize": 2.5,
        "capthick": 0.8
    },
    label="Small cells"
)

ax1.set_ylabel("Recall")
ax1.set_ylim(0, 1.28)

# ============================================================
# FDR: right axis
# ============================================================

ax2 = ax1.twinx()

bars_fdr = ax2.bar(
    x + bar_spacing,
    fdr_mean,
    bar_width,
    yerr=fdr_sem,
    error_kw={
        "elinewidth": 0.8,
        "capsize": 2.5,
        "capthick": 0.8
    },
    color="red",
    label="FDR"
)

ax2.set_ylabel("False discovery rate")
ax2.set_ylim(0, 0.05)

# ============================================================
# Percentage labels
# ============================================================

def add_labels(ax, bars, values, offset):

    for bar, value in zip(bars, values):

        height = bar.get_height()

        if value == 0:
            label = "0%"
        else:
            label = f"{value * 100:.1f}%"

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + offset,
            label,
            ha="center",
            va="bottom",
            fontsize=7
        )

# Large
add_labels(ax1, bars_large, large_mean, offset=0.05)

# Small
add_labels(ax1, bars_small, small_mean, offset=0.13)

# FDR
add_labels(ax2, bars_fdr, fdr_mean, offset=0.005)

# ============================================================
# Replicate markers
# ============================================================

markers = ['o', 'o', 'o', 'o']

for i in range(len(methods)):

    # --------------------------------------------------------
    # Large
    # --------------------------------------------------------

    ax1.scatter(
        x[i] - bar_spacing + jitter,
        large[i] / n_large,
        marker=markers[i],
        facecolors="none",
        edgecolors="black",
        s=20,
        linewidths=0.5,
        zorder=10
    )

    # --------------------------------------------------------
    # Small
    # --------------------------------------------------------

    ax1.scatter(
        x[i] + jitter,
        small[i] / n_small,
        marker=markers[i],
        facecolors="none",
        edgecolors="black",
        s=20,
        linewidths=0.5,
        zorder=10
    )

    # --------------------------------------------------------
    # FDR
    # --------------------------------------------------------

    ax2.scatter(
        x[i] + bar_spacing + jitter,
        empty[i] / n_empty,
        marker=markers[i],
        facecolors="none",
        edgecolors="black",
        s=20,
        linewidths=0.5,
        zorder=10
    )

# ============================================================
# X axis
# ============================================================

ax1.set_xticks(x)

ax1.set_xticklabels(methods, rotation=20, ha='right', fontsize=8)

ax1.set_title('Large/Small/Empty 500/1000/250', fontsize=9, pad=4)

# ============================================================
# Combined legend
# ============================================================

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    frameon=False,
    loc='upper center',
    ncol=3,
    fontsize=8,
    columnspacing=1.0,
    handletextpad=0.4,
    bbox_to_anchor=(0.5, 1.0)
)

# ============================================================
# Layout
# ============================================================

plt.tight_layout()

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["font.size"] = 12

# ============================================================
# Raw results
#
# 每一行 = 一个 replicate
# 每一列 = large / EMPTY / small
#
# 直接按照程序输出填写
# ============================================================

replicates = ["rep1", "rep2", "rep3"]

# ------------------------------------------------------------
# ModuleDrops
# cell_call (col11 == cell)
# ------------------------------------------------------------

module_drops = np.array([[497, 6, 246], [497, 3, 242], [497, 2, 245]], dtype=float)

# ------------------------------------------------------------
# EmptyDrops (non-rRNA)
# padj_nonrrna (col18 < 0.01)
# ------------------------------------------------------------

emptydrops_nonrrna = np.array([[18, 0, 1], [18, 0, 0], [19, 0, 1]], dtype=float)

# ------------------------------------------------------------
# EmptyDrops
# padj_total (col20 < 0.01)
# ------------------------------------------------------------

emptydrops = np.array([[474, 1, 240], [474, 1, 233], [474, 1, 231]], dtype=float)

# ------------------------------------------------------------
# Kneedle
# ------------------------------------------------------------

kneedle = np.array([[60, 0, 0], [60, 0, 0], [60, 0, 0]], dtype=float)

# ============================================================
# Combine into the format used by plotting code
# ============================================================

# Row = method
# Column = replicate

methods = ['ModuleDrops', 'Knee', 'EmptyDrops', 'EmptyDrops\n(non-rRNA)']

large = np.array([module_drops[:, 0], kneedle[:, 0], emptydrops[:, 0], emptydrops_nonrrna[:, 0]])

small = np.array([module_drops[:, 2], kneedle[:, 1], emptydrops[:, 2], emptydrops_nonrrna[:, 2]])

empty = np.array([module_drops[:, 1], kneedle[:, 2], emptydrops[:, 1], emptydrops_nonrrna[:, 1]])

n_large = 500
n_small = 250
n_empty = 1000

# ============================================================
# Mean ± SEM
# ============================================================

# Large cells
large_mean = large.mean(axis=1) / n_large
large_sem = large.std(axis=1, ddof=1) / np.sqrt(large.shape[1]) / n_large

# Small cells
small_mean = small.mean(axis=1) / n_small
small_sem = small.std(axis=1, ddof=1) / np.sqrt(small.shape[1]) / n_small

# Empty droplets / FDR
fdr_mean = empty.mean(axis=1) / n_empty
fdr_sem = empty.std(axis=1, ddof=1) / np.sqrt(empty.shape[1]) / n_empty

# ============================================================
# Plot parameters
# ============================================================

# Method-to-method spacing
x = np.arange(len(methods)) * 0.5

# Width of each bar
bar_width = 0.10

# Distance between centers of the 3 bars
bar_spacing = 0.105

# Replicate jitter
jitter = np.array([-0.01, 0, 0.01])

# ============================================================
# Figure
# ============================================================

fig, ax1 = plt.subplots(figsize=(4.6, 5))

# ============================================================
# Recall: Large cells
# ============================================================

bars_large = ax1.bar(
    x - bar_spacing,
    large_mean,
    bar_width,
    yerr=large_sem,
    error_kw={
        "elinewidth": 0.8,
        "capsize": 2.5,
        "capthick": 0.8
    },
    label="Large cells"
)

# ============================================================
# Recall: Small cells
# ============================================================

bars_small = ax1.bar(
    x,
    small_mean,
    bar_width,
    yerr=small_sem,
    error_kw={
        "elinewidth": 0.8,
        "capsize": 2.5,
        "capthick": 0.8
    },
    label="Small cells"
)

ax1.set_ylabel("Recall")
ax1.set_ylim(0, 1.28)

# ============================================================
# FDR: right axis
# ============================================================

ax2 = ax1.twinx()

bars_fdr = ax2.bar(
    x + bar_spacing,
    fdr_mean,
    bar_width,
    yerr=fdr_sem,
    error_kw={
        "elinewidth": 0.5,
        "capsize": 2.5,
        "capthick": 0.8
    },
    color="red",
    label="FDR"
)

ax2.set_ylabel("False discovery rate")
ax2.set_ylim(0, 0.05)

# ============================================================
# Percentage labels
# ============================================================

def add_labels(ax, bars, values, offset):

    for bar, value in zip(bars, values):

        height = bar.get_height()

        if value == 0:
            label = "0%"
        else:
            label = f"{value * 100:.1f}%"

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + offset,
            label,
            ha="center",
            va="bottom",
            fontsize=7
        )

# Large
add_labels(ax1, bars_large, large_mean, offset=0.05)

# Small
add_labels(ax1, bars_small, small_mean, offset=0.13)

# FDR
add_labels(ax2, bars_fdr, fdr_mean, offset=0.005)

# ============================================================
# Replicate markers
# ============================================================

markers = ['o', 'o', 'o', 'o']

for i in range(len(methods)):

    # --------------------------------------------------------
    # Large
    # --------------------------------------------------------

    ax1.scatter(
        x[i] - bar_spacing + jitter,
        large[i] / n_large,
        marker=markers[i],
        facecolors="none",
        edgecolors="black",
        s=20,
        linewidths=0.5,
        zorder=10
    )

    # --------------------------------------------------------
    # Small
    # --------------------------------------------------------

    ax1.scatter(
        x[i] + jitter,
        small[i] / n_small,
        marker=markers[i],
        facecolors="none",
        edgecolors="black",
        s=20,
        linewidths=0.5,
        zorder=10
    )

    # --------------------------------------------------------
    # FDR
    # --------------------------------------------------------

    ax2.scatter(
        x[i] + bar_spacing + jitter,
        empty[i] / n_empty,
        marker=markers[i],
        facecolors="none",
        edgecolors="black",
        s=20,
        linewidths=0.5,
        zorder=10
    )

# ============================================================
# X axis
# ============================================================

ax1.set_xticks(x)

ax1.set_xticklabels(methods, rotation=20, ha='right', fontsize=8)

ax1.set_title('Large/Small/Empty 500/250/1000', fontsize=9, pad=4)

# ============================================================
# Combined legend
# ============================================================

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    frameon=False,
    loc='upper center',
    ncol=3,
    fontsize=8,
    columnspacing=1.0,
    handletextpad=0.4,
    bbox_to_anchor=(0.5, 1.0)
)

# ============================================================
# Layout
# ============================================================

plt.tight_layout()

plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

# ============================================================
# Observed vs nominal FDR (calibration, EmptyDrops Fig. 1b style)
#
# 3 subsampling scenarios x 3 replicates (no replicate is dropped)
#
# observed FDR = (#EMPTY called) / (#called)
# ============================================================

BASE = "/Users/apple/Downloads/benchmarks/es"
NOMINAL = [0.001, 0.002, 0.005, 0.01]

SCENARIO_FILES = {
    "500/500/500": [
        "test_555_rep1.barcodes.tsv",
        "test_555_rep2.barcodes.tsv",
        "test_555_rep3.barcodes.tsv",
    ],
    "500/1000/250": [
        "test_5125_rep1.barcodes.tsv",
        "test_5125_rep2.barcodes.tsv",
        "test_5125_rep3.barcodes.tsv",
    ],
    "500/250/1000": [
        "test_5251_rep1.barcodes.tsv",
        "test_5251_rep2.barcodes.tsv",
        "test_5251_rep3.barcodes.tsv",
    ],
}

SCENARIOS = list(SCENARIO_FILES)
C_SCEN = ["#3C5488", "#4DBBD5", "#E64B35"]

def observed_fdr(path, col, alpha):

    df = pd.read_csv(
        path,
        sep="\t",
        usecols=["barcode", col],
        dtype={"barcode": str}
    )

    empty = df["barcode"].astype(str).str.startswith("EMPTY").values
    called = df[col].values <= alpha

    if not called.any():
        return np.nan

    return float((empty & called).sum() / called.sum())

fig, ax = plt.subplots(figsize=(6.2, 3.8))
summary = {}

for si, label in enumerate(SCENARIOS):

    files = [f"{BASE}/{f}" for f in SCENARIO_FILES[label]]
    xs, ys = [], []

    for ai, alpha in enumerate(NOMINAL):

        vals = np.array(
            [observed_fdr(f, "padj_combined", alpha) for f in files],
            dtype=float
        )
        vals = vals[~np.isnan(vals)]

        xi = ai + 1 + (si - 1) * 0.22
        jit = np.random.default_rng(0).uniform(-0.045, 0.045, len(vals))

        xs.extend((xi + jit).tolist())
        ys.extend(vals.tolist())

        summary[(label, alpha)] = (vals.mean(), vals.std(ddof=1) / np.sqrt(len(vals)))

        ax.errorbar(
            xi,
            vals.mean(),
            yerr=vals.std(ddof=1) / np.sqrt(len(vals)),
            fmt="o",
            ms=4.5,
            color=C_SCEN[si],
            capsize=2.2,
            elinewidth=0.8,
            capthick=0.8,
            zorder=4
        )

    # single rasterised artist per scenario (heavy point layer)
    ax.plot(
        xs,
        ys,
        "o",
        ms=3.2,
        mfc="none",
        mec=C_SCEN[si],
        mew=0.6,
        alpha=0.75,
        zorder=3,
        rasterized=True,
        label=label
    )

# nominal reference levels (short dotted segment per group)
for i, alpha in enumerate(NOMINAL):

    ax.plot(
        [i + 1 - 0.34, i + 1 + 0.34],
        [alpha, alpha],
        ls=":",
        lw=1.0,
        color="0.45",
        zorder=1
    )

    ax.text(
        i + 1 + 0.40,
        alpha,
        f"{alpha * 100:g}%",
        fontsize=7,
        color="0.35",
        va="center",
        ha="left"
    )

ax.set_xlim(0.55, 5.0)
ax.set_ylim(0, 0.0115)
ax.set_xticks(range(1, 5))
ax.set_xticklabels([f"{a * 100:g}%" for a in NOMINAL])
ax.set_yticks(np.arange(0, 0.0111, 0.002))
ax.set_yticklabels([f"{v * 100:g}%" for v in np.arange(0, 0.0111, 0.002)])
ax.set_xlabel("Nominal FDR")
ax.set_ylabel("Observed FDR")

handles = [Line2D([], [], marker='o', ls='', ms=4.5, color=c, label=l) for c, l in zip(C_SCEN, SCENARIOS)]

ax.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    ncol=3,
    fontsize=8,
    frameon=False,
    handletextpad=0.3,
    columnspacing=1.4
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(labelsize=8, width=0.8, length=3, direction="out")

for s in ("left", "bottom"):
    ax.spines[s].set_linewidth(0.8)

fig.tight_layout()
plt.show()

# observed FDR (%), mean +/- SEM, n = 3
print()
for label in SCENARIOS:
    cells = [f"{summary[(label, a)][0] * 100:5.2f} ({summary[(label, a)][1] * 100:4.2f})"
             for a in NOMINAL]
    print(f"{label:14s}", "  ".join(cells))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["font.size"] = 12

# ==========================
# 读取原始 barcode 数据
# ==========================
df = pd.read_csv('/Users/apple/Downloads/benchmarks/es/run1.barcodes.tsv', sep='\t')

# ==========================
# Method 1: call == cell
# ==========================
df_cells_method1 = df[df['call'] == 'cell'].copy()

# ==========================
# Method 2: Kneedle
# ==========================
df_sorted = df.sort_values('rank').reset_index(drop=True)

df_cells_method2 = df_sorted[df_sorted.index < 1275].copy()

# ==========================
# Method 3: padj_non_rrna < 0.01
# ==========================
df_cells_method3 = df[df['padj_non_rrna'].notna() & (df['padj_non_rrna'] < 0.01)].copy()

# ==========================
# Method 4: padj_gene < 0.01
# 使用另一个文件的barcode筛选
# ==========================

df_gene = pd.read_csv('/Users/apple/Downloads/benchmarks/es/nonrrna1.barcodes.tsv', sep='\t')

method4_barcodes = df_gene.loc[
    (df_gene['padj_non_rrna'].notna()) &
    (df_gene['padj_non_rrna'] < 0.01),
    'barcode'
]

# 从原始df取数据，保证UMI一致
df_cells_method4 = df[df['barcode'].isin(method4_barcodes)].copy()

# ==========================
# 创建 UMI bins
# ==========================

min_umis = min(
    df_cells_method1['umis'].min(),
    df_cells_method2['umis'].min(),
    df_cells_method3['umis'].min(),
    df_cells_method4['umis'].min()
)

max_umis = max(
    df_cells_method1['umis'].max(),
    df_cells_method2['umis'].max(),
    df_cells_method3['umis'].max(),
    df_cells_method4['umis'].max()
)

bins = np.logspace(np.log10(min_umis), np.log10(max_umis), 30)

# ==========================
# 统计每个method的UMI bin分布
# ==========================

def get_bin_stats(data):

    data = data.copy()

    data['log_umi'] = np.log10(data['umis'])

    data['umi_bin'] = pd.cut(
        data['umis'],
        bins=bins,
        include_lowest=True
    )

    stats = data.groupby(
        'umi_bin',
        observed=False
    ).agg(
        cell_count=('barcode','count')
    ).reset_index()

    return stats

bin_stats1 = get_bin_stats(df_cells_method1)

bin_stats2 = get_bin_stats(df_cells_method2)

bin_stats3 = get_bin_stats(df_cells_method3)

bin_stats4 = get_bin_stats(df_cells_method4)

# ==========================
# x轴(log10 bin)
# ==========================

x_step = np.log10(bins)[:-1]

# ==========================
# 绘图
# ==========================

fig, ax = plt.subplots(figsize=(8, 6))

# Method 1
y1 = bin_stats1['cell_count'].values

ax.step(x_step, y1, where='pre', linewidth=1, color='#2E86C1', label='ModuleDrops')

# Method 2
y2 = bin_stats2['cell_count'].values

ax.step(x_step, y2, where='pre', linewidth=1, color='#E74C3C', label='Knee')

# Method 3
y3 = bin_stats3['cell_count'].values

ax.step(x_step, y3, where='pre', linewidth=1, color='#28B463', label='EmptyDrops total rRNA')

# Method 4
y4 = bin_stats4['cell_count'].values

ax.step(x_step, y4, where='pre', linewidth=1, color='#8E44AD', label='EmptyDrops non-rRNA')

# ==========================
# 图形设置
# ==========================

ax.set_xlabel('log10(UMI per barcode)', fontsize=18)

ax.set_ylabel('Number of Cells', fontsize=18)

ax.set_title('Comparison of Cell Calling Methods by UMI Distribution', fontsize=16)

ax.set_xlim(x_step.min(), x_step.max())

ax.tick_params(axis='both', labelsize=16)

# legend放最后
ax.legend(fontsize=13)

# ==========================
# 统计信息
# ==========================

methods = [
    ('ModuleDrops', df_cells_method1),
    ('Knee', df_cells_method2),
    ('EmptyDrops non-rRNA', df_cells_method3),
    ('EmptyDrops total RNA', df_cells_method4)
]

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["font.size"] = 12

# ============================================================
# 1. Read data
# ============================================================

file = "/Users/apple/Downloads/benchmarks/es/test_555_rep1.barcodes.tsv"

df = pd.read_csv(file, sep='\t')

print("Total barcodes:", len(df))

# ============================================================
# 2. Keep only call == cell
# ============================================================

df = df[df['call'].astype(str).str.lower().eq('cell')].copy()

print("Cell barcodes:", len(df))

# ============================================================
# 3. P-value columns
# ============================================================

pcols = {'rRNA%': 'pvalue_rrnapct', 'KM': 'pvalue_ko', 'total rRNA': 'pvalue_total_rrna'}

for col in pcols.values():

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    df[col] = df[col].clip(
        lower=0,
        upper=1
    )

# ============================================================
# 4. Convert P to -log10(P)
# ============================================================

eps = np.finfo(float).tiny

for name, col in pcols.items():

    df[f"logp_{name}"] = -np.log10(
        df[col].clip(lower=eps)
    )

# ============================================================
# 5. Pairwise combinations
# ============================================================

pairs = [
    (
        "rRNA%",
        "KM",
        "rRNA% vs KM"
    ),
    (
        "rRNA%",
        "total rRNA",
        "rRNA% vs total rRNA"
    ),
    (
        "KM",
        "total rRNA",
        "KM vs total rRNA"
    )
]

# ============================================================
# 6. Figure
# ============================================================

fig = plt.figure(figsize=(12, 3.5))

gs = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, 0.06], wspace=0.35)

axes = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[0, 2])]

cax = fig.add_subplot(gs[0, 3])

# ============================================================
# Plot
# ============================================================

hb = None

for ax, (xname, yname, title) in zip(
    axes,
    pairs
):

    x = df[f"logp_{xname}"].to_numpy()
    y = df[f"logp_{yname}"].to_numpy()

    valid = (
        np.isfinite(x) &
        np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    rho, p_corr = spearmanr(x, y)

    hb = ax.hexbin(
        x,
        y,
        gridsize=45,
        mincnt=1,
        bins="log",
        linewidths=0.15
    )

    threshold = 2

    ax.axvline(
        threshold,
        linestyle="--",
        linewidth=0.8
    )

    ax.axhline(
        threshold,
        linestyle="--",
        linewidth=0.8
    )

    ax.set_xlabel(
        f"$-\\log_{{10}}(P)$ — {xname}",
        fontsize=11
    )

    ax.set_ylabel(
        f"$-\\log_{{10}}(P)$ — {yname}",
        fontsize=11
    )

    ax.set_title(
        title,
        fontsize=12
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        labelsize=9
    )

# ============================================================
# Shared colorbar
# ============================================================

cbar = fig.colorbar(hb, cax=cax)

cbar.set_label('Barcode count per hexbin', fontsize=10)

cbar.ax.tick_params(labelsize=9)

plt.show()

# ============================================================
# Layout
# ============================================================

fig.suptitle('Relationship between independent tests', fontsize=13, y=1.02)

fig.subplots_adjust(left=0.07, right=0.96, bottom=0.16, top=0.88, wspace=0.35)

plt.show()
plt.close(fig)
